# Week 4 Day-2: Random Forest + GridSearchCV Comparison
This notebook compares untuned vs tuned Random Forest on Iris.

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from data_loader import load_data
from evaluate import evaluate_model
from random_forest_model import build_random_forest
from tune_random_forest import tune_random_forest

OUTPUT_DIR = PROJECT_ROOT / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
X_train, X_test, y_train, y_test, class_names = load_data(test_size=0.2, random_state=42)
X_train.shape, X_test.shape

In [ ]:
rf_default = build_random_forest(random_state=42)
rf_default.fit(X_train, y_train)
untuned_metrics, _ = evaluate_model('random_forest', rf_default, X_test, y_test)
untuned_metrics

In [ ]:
best_model, best_params, cv_results_df = tune_random_forest(X_train, y_train, random_state=42)
tuned_metrics, _ = evaluate_model('rf_tuned', best_model, X_test, y_test)
best_params

In [ ]:
comparison_df = pd.DataFrame([
    {
        'model': 'random_forest',
        'accuracy': untuned_metrics['accuracy'],
        'precision_macro': untuned_metrics['precision_macro'],
        'recall_macro': untuned_metrics['recall_macro'],
        'f1_macro': untuned_metrics['f1_macro'],
    },
    {
        'model': 'rf_tuned',
        'accuracy': tuned_metrics['accuracy'],
        'precision_macro': tuned_metrics['precision_macro'],
        'recall_macro': tuned_metrics['recall_macro'],
        'f1_macro': tuned_metrics['f1_macro'],
    },
])
comparison_df

In [ ]:
top10 = cv_results_df[[
    'params',
    'mean_test_score',
    'std_test_score',
]].head(10)
top10

In [ ]:
plot_df = cv_results_df[['param_rf__n_estimators', 'mean_test_score']].copy()
plot_df['param_rf__n_estimators'] = plot_df['param_rf__n_estimators'].astype(int)
agg = plot_df.groupby('param_rf__n_estimators', as_index=False)['mean_test_score'].mean().sort_values('param_rf__n_estimators')

plt.figure(figsize=(7, 4))
plt.plot(agg['param_rf__n_estimators'], agg['mean_test_score'], marker='o')
plt.title('Mean CV f1_macro vs n_estimators')
plt.xlabel('n_estimators')
plt.ylabel('mean_test_score')
plt.grid(alpha=0.3)
plt.show()

In [ ]:
cv_results_df.head(50).to_csv(OUTPUT_DIR / 'gridsearch_results.csv', index=False)
(OUTPUT_DIR / 'best_params.json').write_text(json.dumps(best_params, indent=2), encoding='utf-8')
OUTPUT_DIR / 'gridsearch_results.csv', OUTPUT_DIR / 'best_params.json'